2026-07-08 - N_CMA = 6 -> downscale.

state: 3-CMA pool (TOR/MTL/VAN, 75-84, int. only REMl, Psi post-def) banked in saves_eod_2026-06-25. flat "Before" panel is done (truth SD 0.447 vs recovered flat 0). DGP patch written but not yet spent on a city. 3 new CMAs (OTT/CAL/QC) geography done and shapefiles on drive.

do first: EE temp exports for OTT/CAL/QC. Verify each asset feature count = DA Count (2045/1898/1317) in EE code editor before running export, then submit using 9.8 code. should be ~15 min each

once temp lands:

1. two un-run sec 8 fixes: compute_da_mmt feeds 5-vec into 25-min cb_template -> rebuild onebasis from attr(cb_template,"argvar",), predict through reduced basis and diag(1e-8,4) -> diag(1e-8,5).

2. TOR/MTL/VAN are pre-patch, new 3 are post-match. so rebuild all six on v2

3. per new CMA: substrate v2 -> sim -> 150-DA silver stage1 -> reduce. Target n_cma=6

4. CMA-level PC predictors: CMA-mean of 17 vuln vars -> PCA -> PC1-3. stage2 with PCs (n_cma>4 opens) -> downscale Toronto

5. downscaled per-DA vuln vs truth 1+0.4*F1+0.2*F3.

also standing is DRAC full-tor SLURM never drafted. fallback if EE caps

restore one tarball (06-25), self contained. lib cache -> reattach stack -> load() pilot (DGP fns) -> readRDS 06-25 keepers. then retype the 4: build_crossbasis, make_strata_A/B, fit_da_pca

should be: stage2 coef len5 [0.945,-1.986,-0.081,-4.688,7.110]. mtl 6504x765, van 3575x765. 4 fns TRUE

In [ ]:
DRIVE <- "/content/drive/MyDrive/thesis/dlnm-pilot"

system(sprintf("cd /content && cp %s/r_library.tar.gz . && tar -xzf r_library.tar.gz", DRIVE))
.libPaths(c("/content/site-library", .libPaths()))
suppressPackageStartupMessages({
  library(dlnm); library(gnm); library(mixmeta); library(splines)
  library(sf); library(data.table); library(exactextractr); library(terra)
  library(ggplot2); library(viridis); library(lubridate); library(MASS)
})

system(sprintf("cd /content && cp %s/saves_pilot_2026-06-03.tar.gz . && tar -xzf saves_pilot_2026-06-03.tar.gz", DRIVE))
load("/content/saves/pilot_session.RData")

system(sprintf("cd /content && cp %s/saves_eod_2026-06-25.tar.gz . && tar -xzf saves_eod_2026-06-25.tar.gz", DRIVE))
EOD <- "/content/saves_eod"
mtl                 <- readRDS(file.path(EOD, "mtl_substrate.rds"))
van                 <- readRDS(file.path(EOD, "van_substrate.rds"))
cma_age_data_mtlvan <- readRDS(file.path(EOD, "cma_age_data_mtlvan.rds"))
fit_stage1          <- readRDS(file.path(EOD, "fn_fit_stage1.rds"))
qaic                <- readRDS(file.path(EOD, "fn_qaic.rds"))
reduce_fit          <- readRDS(file.path(EOD, "fn_reduce_fit.rds"))
red_tor     <- readRDS(file.path(EOD, "red_tor.rds"))
red_mtl_fit <- readRDS(file.path(EOD, "red_mtl_fit.rds"))
red_van_fit <- readRDS(file.path(EOD, "red_van_fit.rds"))
reduced_tor <- readRDS(file.path(EOD, "reduced_tor.rds"))
reduced_mtl <- readRDS(file.path(EOD, "reduced_mtl.rds"))
reduced_van <- readRDS(file.path(EOD, "reduced_van.rds"))
reduced_df  <- readRDS(file.path(EOD, "reduced_df.rds"))
vcov_list   <- readRDS(file.path(EOD, "vcov_list.rds"))
stage2      <- readRDS(file.path(EOD, "stage2.rds"))

build_crossbasis <- function(T_series, lag_max = 21) {
  T_knots <- quantile(T_series, probs = c(0.10, 0.75, 0.90), na.rm = TRUE)
  cb <- crossbasis(
    x = T_series, lag = lag_max,
    argvar = list(fun = "bs", degree = 2, knots = T_knots),
    arglag = list(fun = "ns", knots = logknots(lag_max, nk = 3))
  )
  stopifnot(attr(cb, "argvar")$fun == "bs")
  stopifnot(attr(cb, "argvar")$degree == 2)
  cb
}
make_strata_A <- function(DA_id, date) factor(paste(DA_id, year(date), month(date), sep = "_"))
make_strata_B <- function(DA_id, date) factor(paste(DA_id, year(date), month(date), wday(date), sep = "_"))

fit_da_pca <- function(Z_matrix, da_ids, verbose = TRUE) {
  pca <- prcomp(Z_matrix, scale. = TRUE)
  cum_var <- summary(pca)$importance["Cumulative Proportion", 3]
  da_scores <- data.table(DAUID = da_ids, PC1 = pca$x[,1], PC2 = pca$x[,2], PC3 = pca$x[,3])
  if (verbose) cat(sprintf("PCA: cum var first 3 = %.1f%%\n", 100*cum_var))
  stopifnot(cum_var > 0.5)
  list(pca = pca, scores = da_scores, var_explained = summary(pca)$importance["Proportion of Variance", 1:3])
}

cat("crossbasis:", exists("crossbasis"), " fread:", exists("fread"), "\n")
cat("stage2 coef len:", length(coef(stage2)), "\n")
print(round(coef(stage2), 3))
cat("mtl temp_mat:", paste(dim(mtl$temp_mat), collapse="x"),
    " van temp_mat:", paste(dim(van$temp_mat), collapse="x"), "\n")
cat("fns:", exists("build_crossbasis"), exists("make_strata_A"),
    exists("make_strata_B"), exists("fit_da_pca"), "\n")

crossbasis: TRUE  fread: TRUE 
stage2 coef len: 5 
theta1 theta2 theta3 theta4 theta5 
 0.945 -1.986 -0.081 -4.688  7.110 
mtl temp_mat: 6504x765  van temp_mat: 3573x765 
fns: TRUE TRUE TRUE TRUE 


probinf v2 before anything. does  build_city_sim_substrate_ve exist? and does its body carry per-CMA per-factor offset?

In [ ]:
if (exists("build_city_sim_substrate_v2")) {
  cat("build_city_sim_substrate_v2: exists\n")
  body_txt <- paste(deparse(body(build_city_sim_substrate_v2)), collapse = "\n")
  cat("  has 'offset' term:", grepl("offset", body_txt, ignore.case = TRUE), "\n") # stale load () could wear v2 name w/ flat guts
  cat("  has 'cma' term:   ", grepl("cma",    body_txt, ignore.case = TRUE), "\n")
  cat("  body lines:", length(deparse(body(build_city_sim_substrate_v2))), "\n")
} else {
  cat("build_city_sim_substrate_v2: missing\n")
}
cat("simulate_counts exists:", exists("simulate_counts"), "\n")

build_city_sim_substrate_v2: missing
simulate_counts exists: TRUE 


In [ ]:
build_city_sim_substrate_v2 <- function(temp_dt, da_age_city, L, seed, mu_sd = 0.6) {
  set.seed(seed)
  da_age_c <- as.data.table(da_age_city)[total > 0]
  temp_wide <- dcast(temp_dt, DAUID ~ date, value.var = "tmean_C")
  common <- intersect(da_age_c$ALT_GEO_CODE, temp_wide$DAUID)
  da_age_c  <- da_age_c[ALT_GEO_CODE %in% common]
  temp_wide <- temp_wide[DAUID %in% common][match(da_age_c$ALT_GEO_CODE, DAUID), ]
  temp_mat  <- as.matrix(temp_wide[, -1]); rownames(temp_mat) <- temp_wide$DAUID

  dead <- which(rowSums(is.na(temp_mat)) == ncol(temp_mat))
  if (length(dead)) {
    keep <- setdiff(seq_len(nrow(temp_mat)), dead)
    temp_mat <- temp_mat[keep, ]; da_age_c <- da_age_c[keep, ]
  }

  n  <- nrow(da_age_c)
  mu <- rnorm(3, mean = 0, sd = mu_sd)              # per-CMA per-factor offset (principle 4)
  tf <- data.table(DAUID = da_age_c$ALT_GEO_CODE, da_idx = 1:n,
                   F1 = rnorm(n, mu[1], 1),
                   F2 = rnorm(n, mu[2], 1),
                   F3 = rnorm(n, mu[3], 1))
  Zc <- as.matrix(tf[, .(F1,F2,F3)]) %*% t(L) + matrix(rnorm(n*17, sd = 0.3), nrow = n)
  colnames(Zc) <- paste0("vuln", sprintf("%02d", 1:17))
  da_age_c[, da_idx := 1:n]
  list(temp_mat = temp_mat, truth_factors = tf, Z = Zc, da_age = da_age_c,
       n_da = n, mu = mu)
}
cat("v2 defined:", exists("build_city_sim_substrate_v2"), "\n")

v2 defined: TRUE 


fix compute_da_mmt. reduced curve is 5-dim and old code predicts it through the 25-dim cb_template -> dim. mismatch.

rebuild onebasis from attr(cb_template), "argvar") = the stored bs/degree/knots spec crossreduce used, predict the 5-vec through the 5-dim basis.

In [ ]:
compute_da_mmt <- function(da_theta_row, cb_template, temp_range_da, cma_median) {
  av <- attr(cb_template, "argvar")
  red_basis <- onebasis(temp_range_da, fun = av$fun, degree = av$degree, knots = av$knots)
  pred1 <- crosspred(basis = red_basis,
                     coef  = da_theta_row,
                     vcov  = diag(1e-8, 5),
                     at    = temp_range_da,
                     cen   = cma_median)
  pred1$predvar[which.min(pred1$allfit)]
}

# smoke test on the banked pool + Toronto template
test_theta <- as.numeric(coef(stage2))
test_range <- seq(min(reduced_tor$reduced_obj$predvar),
                  max(reduced_tor$reduced_obj$predvar), length.out = 50)
mmt_test <- compute_da_mmt(test_theta, red_tor$cb_template, test_range, reduced_tor$cen)
cat("MMT test:", round(mmt_test, 1), " in range:",
    round(min(test_range),1), "-", round(max(test_range),1), "\n")
cat("length coef fed:", length(test_theta), " basis dim: 5 (reduced)\n")

MMT test: 25.8  in range: 4 - 30 
length coef fed: 5  basis dim: 5 (reduced)


MMT test: 25.8  in range: 4 - 30
length coef fed: 5  basis dim: 5 (reduced)

---

compute_da_mmt fixed. 5-vec now predicts through the 5-dim reduced basis, diag 4->5. 25.8 in range 4-30.

EE finished. Per city: read, check unique DA count = 2045/1898/1317. rows = DAx765 date span warm season 2015-19. NA count.

In [ ]:
new_csvs <- c(ottawa  = "ottawa_daymet_2015_2019.csv",
              calgary = "calgary_daymet_2015_2019.csv",
              quebec  = "quebec_daymet_2015_2019.csv")
expected_da <- c(ottawa = 2045, calgary = 1898, quebec = 1317)

for (city in names(new_csvs)) {
  path <- file.path(DRIVE, new_csvs[city])
  if (!file.exists(path)) { cat(sprintf("%-8s MISSING FILE\n", city)); next }
  d <- fread(path, select = c("DAUID","date","tmax","tmin"))
  d[, DAUID := as.character(DAUID)]
  d[, date  := as.Date(date)]
  n_da_actual <- uniqueN(d$DAUID) # count distinct DAs to catch truncated or partial export because row count could look plausible while DAs are silently missing
  na_temp     <- sum(is.na(d$tmax) | is.na(d$tmin))
  cat(sprintf("%-8s rows:%8d  unique_DA:%5d / %5d  %s  dates:%s..%s  NA:%d\n",
              city, nrow(d), n_da_actual, expected_da[city],
              if (n_da_actual == expected_da[city]) "OK" else "MISMATCH-STOP",
              as.character(min(d$date)), as.character(max(d$date)), na_temp))
}

ottawa   rows: 1564425  unique_DA: 2045 /  2045  OK  dates:2015-05-01..2019-09-30  NA:0
calgary  rows: 1451970  unique_DA: 1898 /  1898  OK  dates:2015-05-01..2019-09-30  NA:0
quebec   rows: 1007505  unique_DA: 1317 /  1317  OK  dates:2015-05-01..2019-09-30  NA:765


ottawa   rows: 1564425  unique_DA: 2045 /  2045  OK  dates:2015-05-01..2019-09-30  NA:0
calgary  rows: 1451970  unique_DA: 1898 /  1898  OK  dates:2015-05-01..2019-09-30  NA:0
quebec   rows: 1007505  unique_DA: 1317 /  1317  OK  dates:2015-05-01..2019-09-30  NA:765

---

all 3 verified: DA count exact, rows=DAx765, dates clean. quebec NA:765 = interestingly one all-water DA (765 = 1 DA x 765 days), not scattered like MTL
s 28. St Lawrence puts one polygon fully in-river. Calgary's 0 since it is landlocked, ottawa 0—river is the boundary and DAs drawn to banks not channel... interesting!).

rebuild TOR on v2, 1st of 6. v1 in env is iid/flat; v2 draws TOR its own mu offset (seed42+prefix 35 = 77). watch that mu triple must be nonzero + distinct because that offset is the patch, mu~0 = accidental v1. n_da ~7682 (because 12 water already gone from this CSV), aligned, NA 0.

In [ ]:
tor_daymet <- read_daymet(daymet_csv)
tor_v2 <- build_city_sim_substrate_v2(tor_daymet, da_age, L, seed = 42 + 35)

cat("n_da:", tor_v2$n_da, "\n")
cat("temp_mat:", paste(dim(tor_v2$temp_mat), collapse=" x "),
    " align:", all(rownames(tor_v2$temp_mat) == tor_v2$truth_factors$DAUID),
    " NA:", sum(is.na(tor_v2$temp_mat)), "\n")
cat("mu:", paste(round(tor_v2$mu, 3), collapse=" "), "\n") # must be nonzero and distinct
cat("F means:", paste(round(c(mean(tor_v2$truth_factors$F1),
                              mean(tor_v2$truth_factors$F2),
                              mean(tor_v2$truth_factors$F3)), 3), collapse=" "), "\n")

ERROR: Error in read_daymet(daymet_csv): could not find function "read_daymet"


Error in read_daymet(daymet_csv): could not find function "read_daymet"
Traceback:

---

read_daymet doesn't exist in envrionment. read_daymet needs to join the retype list in the restore cell

In [ ]:
read_daymet <- function(csv) {
  d <- fread(csv, select = c("DAUID","date","tmax","tmin"))
  d[, `:=`(DAUID = as.character(DAUID), date = as.Date(date), tmean_C = (tmax+tmin)/2)]
  d[]
}
cat("read_daymet defined:", exists("read_daymet"), "\n")

read_daymet defined: TRUE 


read_daymet defined: TRUE

---

re-run previous block

In [ ]:
tor_daymet <- read_daymet(daymet_csv)
tor_v2 <- build_city_sim_substrate_v2(tor_daymet, da_age, L, seed = 42 + 35)

cat("n_da:", tor_v2$n_da, "\n")
cat("temp_mat:", paste(dim(tor_v2$temp_mat), collapse=" x "),
    " align:", all(rownames(tor_v2$temp_mat) == tor_v2$truth_factors$DAUID),
    " NA:", sum(is.na(tor_v2$temp_mat)), "\n")
cat("mu:", paste(round(tor_v2$mu, 3), collapse=" "), "\n") # must be nonzero and distinct
cat("F means:", paste(round(c(mean(tor_v2$truth_factors$F1),
                              mean(tor_v2$truth_factors$F2),
                              mean(tor_v2$truth_factors$F3)), 3), collapse=" "), "\n")

n_da: 7682 
temp_mat: 7682 x 765  align: TRUE  NA: 0 
mu: -0.33 0.655 0.384 
F means: -0.329 0.657 0.388 


n_da: 7682
temp_mat: 7682 x 765  align: TRUE  NA: 0
mu: -0.33 0.655 0.384
F means: -0.329 0.657 0.388

---

TOR substrate rebuilt on v2. mu [-0.33, 0.655, 0.384] - all distinct nonzero, patch worked (V1 would be flat 0). F means [-0.329, 0.657, 0.388] land on mu not 0 -> offset propogated into draws. n_da 7682 = 12 water gone, align TRUE< NA 0.

tor sim on v2. same simulate_counts as v1 (189,137), only factors changed - carry TOR's mu now. mmt off new temp_mat. deaths must be NEAR 189k not equal - equal = offset didn't reach counts = suspicious. structure holds if age gradient climbs, ~99% zero.

In [ ]:
mmt_tor_v2 <- apply(tor_v2$temp_mat, 1, function(x) quantile(x, 0.80, na.rm = TRUE))

tor_da_long <- CJ(da_idx = 1:tor_v2$n_da, date = study_dates, age_band = names(annual_rates))
pop_long <- melt(tor_v2$da_age[, .(da_idx, age_0_64, age_65_74, age_75_84, age_85p)],
                 id.vars = "da_idx", variable.name = "age_band", value.name = "pop")
pop_long[, age_band := as.character(age_band)]
tor_da_long <- pop_long[tor_da_long, on = c("da_idx","age_band")]
tor_da_long[, annual_rate := annual_rates[age_band]]
tor_da_long[, lambda0 := pop * annual_rate / 1000 / 365]

tor_sim_v2 <- simulate_counts(tor_v2$temp_mat, tor_da_long, tor_v2$truth_factors, mmt_tor_v2, seed = 42)

cat("da_long rows:", nrow(tor_da_long), " NA pop:", sum(is.na(tor_da_long$pop)), "\n")
cat("total deaths:", sum(tor_sim_v2$n_deaths), " (v1 was 189137)  NA:", sum(is.na(tor_sim_v2$n_deaths)), "\n")
print(tor_sim_v2[, .(deaths = sum(n_deaths)), by = age_band])

Simulating 7682 DAs × 765 days × 4 age bands
Total deaths simulated: 242509 
Mean deaths per DA-day-age: 0.0103 
% zero days: 99 %
da_long rows: 23506920  NA pop: 0 
total deaths: 242509  (v1 was 189137)  NA: 0 
    age_band deaths
      <char>  <int>
1:  age_0_64  12831
2: age_65_74  56122
3: age_75_84  75055
4:   age_85p  98501


Simulating 7682 DAs × 765 days × 4 age bands
Total deaths simulated: 242509
Mean deaths per DA-day-age: 0.0103
% zero days: 99 %
da_long rows: 23506920  NA pop: 0
total deaths: 242509  (v1 was 189137)  NA: 0
    age_band deaths
      <char>  <int>
1:  age_0_64  12831
2: age_65_74  56122
3: age_75_84  75055
4:   age_85p  98501

---

Tor v2 sim: 242,509 deaths vs v1's 189,137. not equal means patch reached the counts (good). rose because mu2=+0.655 is the big draw and cold rides 0.3*F2 - TOR v2 drew high cold-vulnerability, +0.66*0.3 lift across every DA. age gradient climbs 12.8k->56k->75k->98.5k, ~99% zero, NA 0, rows 23.5M. TOR has a city character now (high-F2); that between-city spread is what stage2 slope will read

Tor v2 silver stage 1. 150 DAs seed 42, age 75-84, temp attached in row order, cb built, both variants and qAIC picks. sim underneath is v2 (242k) so counts shifted, qAIC magnitude may drift from v1's 28,979 but A must beat B wide bec. sparse silver starves B's weekday strata.

In [ ]:
tor_v2_fit <- fit_city_sliver(tor_sim_v2, tor_v2$temp_mat, "Toronto", "age_75_84")

r <- tor_v2_fit$res
cat("cross-basis:", paste(dim(tor_v2_fit$cb), collapse=" x "), "\n")
cat("winner:", r$winner_variant, " qAIC A:", round(r$qaic_A,1),
    " B:", round(r$qaic_B,1), " (v1 A was 28979)\n")
cat("coef:", length(r$coef), " vcov:", paste(dim(r$vcov), collapse=" x "),
    " any NA:", any(is.na(r$coef)), "\n")

ERROR: Error in fit_city_sliver(tor_sim_v2, tor_v2$temp_mat, "Toronto", "age_75_84"): could not find function "fit_city_sliver"


Error in fit_city_sliver(tor_sim_v2, tor_v2$temp_mat, "Toronto", "age_75_84"): could not find function "fit_city_sliver"
Traceback:

---

again... 6.4 helper not in the restore, even though fit_city_silver is already defined (in SSOT). RESTORE BLOCK FIX NEEDED! anyway, here it is again

In [ ]:
fit_city_sliver <- function(sim_dt, temp_mat, cma_label, age = "age_75_84", n = 150, seed = 42) {
  set.seed(seed)
  idx <- sample(unique(sim_dt$da_idx), n)
  sl  <- sim_dt[da_idx %in% idx & age_band == age]
  sl[, DA_id := da_idx]
  setorder(sl, da_idx, date)
  tl <- data.table(
    da_idx = rep(idx, each = length(study_dates)),
    date   = rep(study_dates, times = n),
    temp_C = as.vector(t(temp_mat[idx, ]))
  )
  sl <- tl[sl, on = c("da_idx","date")]
  cb <- build_crossbasis(sl$temp_C, lag_max = 21)
  res <- fit_stage1(sl, cb, cma_label, age)
  list(sliver = sl, cb = cb, res = res)
}
cat("fit_city_sliver defined:", exists("fit_city_sliver"), "\n")

fit_city_sliver defined: TRUE 


In [ ]:
tor_v2_fit <- fit_city_sliver(tor_sim_v2, tor_v2$temp_mat, "Toronto", "age_75_84")

r <- tor_v2_fit$res
cat("cross-basis:", paste(dim(tor_v2_fit$cb), collapse=" x "), "\n")
cat("winner:", r$winner_variant, " qAIC A:", round(r$qaic_A,1),
    " B:", round(r$qaic_B,1), " (v1 A was 28979)\n")
cat("coef:", length(r$coef), " vcov:", paste(dim(r$vcov), collapse=" x "),
    " any NA:", any(is.na(r$coef)), "\n")


=== Stage 1: Toronto, age age_75_84 ===
  Variant A: 3750 strata, 114750 rows
  Variant A: FAILED to converge
  Variant B: 26250 strata, 114750 rows
  Variant B: FAILED to converge
  Winner: variant B


ERROR: Error in UseMethod("vcov"): no applicable method for 'vcov' applied to an object of class "NULL"



=== Stage 1: Toronto, age age_75_84 ===
  Variant A: 3750 strata, 114750 rows
  Variant A: FAILED to converge
  Variant B: 26250 strata, 114750 rows
  Variant B: FAILED to converge
  Winner: variant B
Error in UseMethod("vcov"): no applicable method for 'vcov' applied to an object of class "NULL"
Traceback:

1. fit_stage1(sl, cb, cma_label, age)
2. vcov(results[[winner_var]]$fit)
3. .handleSimpleError(function (cnd)
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"),
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "no applicable method for 'vcov' applied to an object of class \"NULL\"",
 .     base::quote(UseMethod("vcov")))

 ---

 both variants failed to converge. the vcov(NULL) error is downstream, fit_stage1 defaults to B when both fail, B$fit is null, vcov(NULL) throws. I flagged thjis in SSOT as fit_stage1 hardening. birth = non-convergence. Why: v2's mu (mu2=+0.66 cold, mu1=-0.33 heat) makes the silver's log-RR surface steeper + lopsided vs v1's flat ~0 factors. 150 DAs too think for gnm IRLS to fit the steeper surface.

retry TOR v2 silver at n=300 (was 150).  wider net = more data to stabilize. if con verges: thinness confirm, and A should win. Double the rows in cross basis.

In [ ]:
tor_v2_fit <- fit_city_sliver(tor_sim_v2, tor_v2$temp_mat, "Toronto", "age_75_84", n = 300)

r <- tor_v2_fit$res
cat("cross-basis:", paste(dim(tor_v2_fit$cb), collapse=" x "), "\n")
cat("winner:", r$winner_variant, " qAIC A:", round(r$qaic_A,1),
    " B:", round(r$qaic_B,1), "\n")
cat("coef:", length(r$coef), " vcov:", paste(dim(r$vcov), collapse=" x "),
    " any NA:", any(is.na(r$coef)), "\n")


=== Stage 1: Toronto, age age_75_84 ===
  Variant A: 7500 strata, 229500 rows
  Variant A: FAILED to converge
  Variant B: 52500 strata, 229500 rows
  Variant B: FAILED to converge
  Winner: variant B


ERROR: Error in UseMethod("vcov"): no applicable method for 'vcov' applied to an object of class "NULL"



=== Stage 1: Toronto, age age_75_84 ===
  Variant A: 7500 strata, 229500 rows
  Variant A: FAILED to converge
  Variant B: 52500 strata, 229500 rows
  Variant B: FAILED to converge
  Winner: variant B
Error in UseMethod("vcov"): no applicable method for 'vcov' applied to an object of class "NULL"
Traceback:

1. fit_stage1(sl, cb, cma_label, age)
2. vcov(results[[winner_var]]$fit)
3. .handleSimpleError(function (cnd)
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"),
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "no applicable method for 'vcov' applied to an object of class \"NULL\"",
 .     base::quote(UseMethod("vcov")))


---

n=300 also failed both variants; problem is structural not thinness. mechanism: v2 mu1=-0.33 shifts F1 mass negative, so 0.4*F1 < -1 for many DAs -> (1+0.4*F1) goes NEGATIVE -> heat lowers mortality in those DAs -> U-curve inverts on part of the city. gnm can't fit one cb to a response that points up AND down across pooled DAs.

DGP fix: additive -> exp modulation in simulate_counts. the old was 1+0.4*F1*heat+...) can go negative -> U-curve inverts -> gnm can't fit. exp(...) always positive, curve stays a U. view live body first, then redefine with the one line swapped. after: grep body for exp( present, (1+0.4 gone.)

this changes every downstream number + recovery truth target becomes exp (0.4*F1+0.2*F3) not additive.

In [ ]:
cat("simulate_counts exists:", exists("simulate_counts"), "\n")
cat("--- current modulation line ---\n")
body_txt <- deparse(body(simulate_counts))
cat(grep("log_rr_mat\\[, j\\]", body_txt, value = TRUE), sep = "\n")

simulate_counts exists: TRUE 
--- current modulation line ---
        log_rr_mat[, j] <- base_j * (1 + 0.4 * F1 * heat_ind + 


redefine simulate_counts exp form. base_j*(1+0*F1*heat+...) -> base_j*exp(0.4*F1*heat+...), rest is all same.

In [ ]:
simulate_counts <- function(temp_mat, da_long, truth_factors, mmt_da, seed = 42) {
  set.seed(seed)
  n_da   <- nrow(temp_mat)
  n_days <- ncol(temp_mat)

  F1 <- truth_factors$F1; F2 <- truth_factors$F2; F3 <- truth_factors$F3
  log_rr_mat <- matrix(0, n_da, n_days)
  for (j in 1:n_days) {
    T_j     <- temp_mat[, j]
    base_j  <- base_log_rr(T_j, mmt_da)
    heat_ind <- pmax(T_j - mmt_da, 0) > 0
    cold_ind <- pmax(mmt_da - T_j, 0) > 0
    log_rr_mat[, j] <- base_j * exp(0.4*F1*heat_ind + 0.3*F2*cold_ind + 0.2*F3) # exp of the factor combination is always positive so base_j keeps its sign for every DA
  }

  log_rr_lagged <- matrix(0, n_da, n_days)
  for (i in 1:n_da) {
    log_rr_lagged[i, ] <- stats::filter(log_rr_mat[i, ], lag_weights, sides = 1)
  }
  log_rr_lagged[is.na(log_rr_lagged)] <- 0

  date_lookup <- data.table(date = study_dates, date_idx = seq_along(study_dates))
  da_long2 <- merge(da_long, date_lookup, by = "date", all.x = TRUE)
  setkey(da_long2, da_idx, date_idx, age_band)

  da_long2[, log_rr   := log_rr_lagged[cbind(da_idx, date_idx)]]
  da_long2[, lambda   := lambda0 * exp(log_rr)]
  da_long2[, n_deaths := rpois(.N, lambda)]

  return(da_long2)
}

bt <- deparse(body(simulate_counts))
cat("exp( present:", any(grepl("exp\\(0.4", bt)), "\n")
cat("(1 + 0.4 gone:", !any(grepl("\\(1 \\+ 0.4", bt)), "\n")

exp( present: TRUE 
(1 + 0.4 gone: TRUE 


exp( present: TRUE
(1 + 0.4 gone: TRUE

---

simulate_counts redefined to exp form, DGP now multiplicative and curve can't invert.

re-sim TOR under exp DGP, same tor_v2 substrate with mu unchanged, seed 42 same mmt, only modulation form changed.

242,509 is dead (additive), exp scales differently, expecting different and likely higher (exp of mean-zero > 1, Jensen). all else holds as before: gradient climb, ~99% zero, NA pop 0.

In [ ]:
tor_sim_v2 <- simulate_counts(tor_v2$temp_mat, tor_da_long, tor_v2$truth_factors, mmt_tor_v2, seed = 42)

cat("total deaths:", sum(tor_sim_v2$n_deaths), " (additive was 242509)  NA:", sum(is.na(tor_sim_v2$n_deaths)), "\n")
cat("da_long rows:", nrow(tor_da_long), " NA pop:", sum(is.na(tor_da_long$pop)), "\n")
print(tor_sim_v2[, .(deaths = sum(n_deaths)), by = age_band])

total deaths: 610216  (additive was 242509)  NA: 0 
da_long rows: 23506920  NA pop: 0 
    age_band deaths
      <char>  <int>
1:  age_0_64  38614
2: age_65_74 161341
3: age_75_84 189439
4:   age_85p 220822


total deaths: 610216  (additive was 242509)  NA: 0
da_long rows: 23506920  NA pop: 0
    age_band deaths
      <char>  <int>
1:  age_0_64  38614
2: age_65_74 161341
3: age_75_84 189439
4:   age_85p 220822

---

exp re-sim TOR: 610,216 vs additive 242,509 = 2.5x. two multiplicative lifts stacking: (1) jensen, e[exp(x)]>1 for mean-zero x, exp modulation no longer avgs to 1 like (1+...) did; (2) mu2=+0.655 pushes cold factor mean positive, exp(0.3*F@) averages well above 1. gradient climbs 38k->161k->189k->221k, NA 0, rows 23.5M. structure sane and DGP intact.

exp inflates baseline mortality (unrealistic level) but pilot validates architecture not calibration, so absolute levels irrelavant.

tor silver n=150 under exp sim, additive failed both variants (Sign-flip inverted curves). exp can't invert so gnm should see one coherent U across all DAs. the tell is that it converges, then A beats B wide, cb 114750x25, coef 25 no NA. if vCOV(null) again = rethink

In [ ]:
tor_v2_fit <- fit_city_sliver(tor_sim_v2, tor_v2$temp_mat, "Toronto", "age_75_84", n = 150)

r <- tor_v2_fit$res
cat("cross-basis:", paste(dim(tor_v2_fit$cb), collapse=" x "), "\n")
cat("winner:", r$winner_variant, " qAIC A:", round(r$qaic_A,1),
    " B:", round(r$qaic_B,1), "\n")
cat("coef:", length(r$coef), " vcov:", paste(dim(r$vcov), collapse=" x "),
    " any NA:", any(is.na(r$coef)), "\n")


=== Stage 1: Toronto, age age_75_84 ===
  Variant A: 3750 strata, 114750 rows
  Variant A: FAILED to converge
  Variant B: 26250 strata, 114750 rows
  Variant B: FAILED to converge
  Winner: variant B


ERROR: Error in UseMethod("vcov"): no applicable method for 'vcov' applied to an object of class "NULL"



=== Stage 1: Toronto, age age_75_84 ===
  Variant A: 3750 strata, 114750 rows
  Variant A: FAILED to converge
  Variant B: 26250 strata, 114750 rows
  Variant B: FAILED to converge
  Winner: variant B
Error in UseMethod("vcov"): no applicable method for 'vcov' applied to an object of class "NULL"
Traceback:

1. fit_stage1(sl, cb, cma_label, age)
2. vcov(results[[winner_var]]$fit)
3. .handleSimpleError(function (cnd)
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"),
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "no applicable method for 'vcov' applied to an object of class \"NULL\"",
 .     base::quote(UseMethod("vcov")))


 ---

 exp silver also failed both variants, diagnosis is wrong. ugh.

 sign-flip was real but not the convergence blocker because removing it changed nothing. We probably should have read gnm's actual error because "Failed to converge" is fit_stage1's wrapper string.